# 3-Model Prediction Comparison

This notebook helps you inspect and compare predictions from:
- Linear Regression
- Decision Tree
- Random Forest

It uses the trained artifacts in `artifacts/models/` and metadata from `artifacts/preprocessors/model_comparison_metadata.json`.

In [ ]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd

from src.models.predict import MultiModelInferenceEngine
from src.utils.config import COMPARE_METADATA_PATH

engine = MultiModelInferenceEngine()
metadata = engine.metadata

print('Best model by RMSE:', metadata.get('best_model', 'N/A'))
print('Available models:', ', '.join(metadata.get('models', {}).keys()))

In [ ]:
def predict_case(
    province: str = 'Phnom Penh',
    temp_min: float = 24.0,
    rain: float = 1.2,
    wind_speed: float = 12.0,
    lat: float = 11.55,
    lon: float = 104.91,
    target_date: date = date.today(),
) -> pd.DataFrame:
    preds = engine.predict_all(
        temp_min=temp_min,
        rain=rain,
        wind_speed=wind_speed,
        lat=lat,
        lon=lon,
        year=target_date.year,
        month=target_date.month,
        day=target_date.day,
        dayofweek=target_date.weekday(),
        province=province,
    )

    rows = []
    model_meta = metadata.get('models', {})
    best_model = metadata.get('best_model')

    for model_key, pred in preds.items():
        metrics = model_meta.get(model_key, {}).get('metrics', {})
        rows.append({
            'model': model_key,
            'predicted_temp_max_c': round(float(pred), 2),
            'rmse': metrics.get('rmse'),
            'mae': metrics.get('mae'),
            'r2': metrics.get('r2'),
            'is_best': model_key == best_model,
        })

    return pd.DataFrame(rows).sort_values('rmse', na_position='last').reset_index(drop=True)

In [ ]:
comparison_df = predict_case(
    province='Phnom Penh',
    temp_min=25.0,
    rain=2.5,
    wind_speed=11.0,
    lat=11.55,
    lon=104.91,
    target_date=date(2026, 4, 6),
)

comparison_df

In [ ]:
metrics_table = []
for model_key, details in metadata.get('models', {}).items():
    m = details.get('metrics', {})
    metrics_table.append({
        'model': model_key,
        'model_type': details.get('model_type'),
        'rmse': m.get('rmse'),
        'mae': m.get('mae'),
        'r2': m.get('r2'),
    })

pd.DataFrame(metrics_table).sort_values('rmse').reset_index(drop=True)